In [2]:
# table4_best_meanpm.py
# 목적:
# - Table 4(mean±sd) 전체 버전(table4_meanpm.csv)과 동일한 "표 구조"를 유지하되,
#   각 (SET, DAG) 조합에서 AUPRC(평균) 기준 '최우수 모델 1개'만 남긴 CSV를 추가로 생성
#
# 입력(모두 ./):
#   results_O_1.csv ... results_O_5.csv
#   results_F_1.csv ... results_F_5.csv
#   results_OF_1.csv ... results_OF_5.csv
#
# 출력:
#   table4_meanpm.csv          : 전체 조합 mean±sd (완전판)
#   table4_best_meanpm.csv     : (SET, DAG)별 best model만 mean±sd (요약판)
#
# 가정(필수 컬럼):
#   SET, DAG, MODEL, K_EDGE, N_FEAT, FEATURE_KEY, AUROC, AUPRC, F1, Brier, ECE

import os
import re
import glob
import pandas as pd

METRICS = ["AUROC", "AUPRC", "F1", "Brier", "ECE"]
GROUP_COLS = ["SET", "DAG", "MODEL", "K_EDGE", "N_FEAT", "FEATURE_KEY"]

def load_results(pattern: str) -> pd.DataFrame:
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No files matched: {pattern}")

    dfs = []
    for p in paths:
        df = pd.read_csv(p)
        m = re.search(r"_(\d+)\.csv$", os.path.basename(p))
        df["SEED_RUN"] = int(m.group(1)) if m else None
        df["SOURCE_FILE"] = os.path.basename(p)
        dfs.append(df)

    out = pd.concat(dfs, ignore_index=True)

    required = GROUP_COLS + METRICS
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    return out

def fmt(mean: float, std: float, digits: int) -> str:
    return f"{mean:.{digits}f} ± {std:.{digits}f}"

def aggregate_meanpm(df_all: pd.DataFrame) -> pd.DataFrame:
    agg = (
        df_all.groupby(GROUP_COLS)[METRICS]
              .agg(["mean", "std", "count"])
              .reset_index()
    )

    # flatten columns
    agg.columns = [
        "_".join([x for x in col if x]) if isinstance(col, tuple) else col
        for col in agg.columns
    ]
    # restore group col names
    for c in GROUP_COLS:
        if c + "_" in agg.columns:
            agg.rename(columns={c + "_": c}, inplace=True)

    out = agg[GROUP_COLS].copy()
    out["N_RUNS"] = agg["AUROC_count"].astype(int)

    out["AUROC (Mean ± SD)"] = [fmt(m, s, 3) for m, s in zip(agg["AUROC_mean"], agg["AUROC_std"])]
    out["AUPRC (Mean ± SD)"] = [fmt(m, s, 3) for m, s in zip(agg["AUPRC_mean"], agg["AUPRC_std"])]
    out["F1-Score (Mean ± SD)"] = [fmt(m, s, 3) for m, s in zip(agg["F1_mean"], agg["F1_std"])]
    out["Brier (Mean ± SD)"] = [fmt(m, s, 5) for m, s in zip(agg["Brier_mean"], agg["Brier_std"])]
    out["ECE (Mean ± SD)"] = [fmt(m, s, 5) for m, s in zip(agg["ECE_mean"], agg["ECE_std"])]

    # keep helper columns for best selection
    out["_AUPRC_mean_raw"] = agg["AUPRC_mean"]

    # sort similarly to Table 4
    out = out.sort_values(["SET", "DAG", "MODEL", "K_EDGE"]).reset_index(drop=True)
    return out

def select_best_per_set_dag(table4_full: pd.DataFrame) -> pd.DataFrame:
    # (SET, DAG)별 AUPRC_mean 최대 1개 선택
    best = (
        table4_full.sort_values(["SET", "DAG", "_AUPRC_mean_raw"], ascending=[True, True, False])
                  .groupby(["SET", "DAG"], as_index=False)
                  .head(1)
                  .copy()
    )

    # 표 구조는 동일하게: helper 제거 후 정렬
    best = best.drop(columns=["_AUPRC_mean_raw"])
    best = best.sort_values(["SET", "DAG"]).reset_index(drop=True)
    return best

def finalize_full(table4_full: pd.DataFrame) -> pd.DataFrame:
    # full도 helper 제거해서 저장
    return table4_full.drop(columns=["_AUPRC_mean_raw"])

def main():
    df_O  = load_results("./results_O_*.csv")
    df_F  = load_results("./results_F_*.csv")
    df_OF = load_results("./results_OF_*.csv")

    df_all = pd.concat([df_O, df_F, df_OF], ignore_index=True)

    table4_full = aggregate_meanpm(df_all)
    table4_best = select_best_per_set_dag(table4_full)

    out_full = "./table4_meanpm.csv"
    out_best = "./table4_best_meanpm.csv"

    finalize_full(table4_full).to_csv(out_full, index=False, encoding="utf-8-sig")
    table4_best.to_csv(out_best, index=False, encoding="utf-8-sig")

    print(f"[OK] Saved: {out_full}  (rows={len(table4_full)})")
    print(f"[OK] Saved: {out_best} (rows={len(table4_best)})")
    print("\nPreview (best):")
    print(table4_best.to_string(index=False))

if __name__ == "__main__":
    main()

[OK] Saved: ./table4_meanpm.csv  (rows=1040)
[OK] Saved: ./table4_best_meanpm.csv (rows=12)

Preview (best):
SET     DAG    MODEL  K_EDGE  N_FEAT      FEATURE_KEY  N_RUNS AUROC (Mean ± SD) AUPRC (Mean ± SD) F1-Score (Mean ± SD) Brier (Mean ± SD)   ECE (Mean ± SD)
  F     GES LightGBM      34      34 b154670abb75bb96       5     0.928 ± 0.002     0.468 ± 0.004        0.453 ± 0.007 0.01690 ± 0.00023 0.01684 ± 0.00039
  F   GOLEM LightGBM      25      25 710761342f601788       5     0.889 ± 0.005     0.350 ± 0.012        0.386 ± 0.027 0.01903 ± 0.00031 0.01880 ± 0.00077
  F NOTEARS  XGBoost       6       6 4219c68b6a956822       5     0.867 ± 0.003     0.247 ± 0.006        0.320 ± 0.022 0.01908 ± 0.00006 0.00969 ± 0.00065
  F      PC LightGBM      21      21 ac0713d3eecfcacd       5     0.906 ± 0.003     0.422 ± 0.004        0.435 ± 0.012 0.01762 ± 0.00014 0.01737 ± 0.00045
  O     GES  XGBoost       0      13 d142655db65fda0d       5     0.936 ± 0.003     0.455 ± 0.004        0.443 ± 0.0